In [ ]:
# Setup, Version check and Common imports

# Python ≥ 3.7 is required
import sys
assert sys.version_info >= (3, 7)


# TensorFlow ≥ 2.8 is required
import tensorflow as tf
from packaging import version

assert version.parse(tf.__version__) >= version.parse("2.8.0")

# Common imports
import numpy as np
import os
import pandas as pd

from tensorflow import keras
from tensorflow.keras import layers

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

print('Python version: ', sys.version_info)
print('TF version: ', tf.__version__)
print('Keras version: ', keras.__version__)
print('GPU is', 'available' if tf.config.list_physical_devices('GPU') else 'NOT AVAILABLE')

**RESNET 50 - Keras Applications**

**1. RESNET50 for Inference**

In [ ]:
# Load RESNET50 from keras applications
# https://keras.io/api/applications/
# https://keras.io/guides/transfer_learning/


# The ResNet-50 Model will be used to predict the category of selected images
# https://keras.io/api/applications/resnet/#resnet50-function

# Load Model with weights trained on Imagenet

from tensorflow.keras import applications
from tensorflow.keras.applications.resnet50 import ResNet50

modelR = ResNet50(weights='imagenet')

In [ ]:
# Load two sample images and resize them
# to match the input required by ResNet

# Try with several images and analyze the outcomes

resize = layers.Resizing(224, 224, crop_to_aspect_ratio=True)

def load_and_resize(path):
    img = keras.utils.load_img(path)
    arr = keras.utils.img_to_array(img)
    arr = tf.expand_dims(arr, axis=0)
    return resize(arr)

Image1 = 'A.jpeg' #@param {type:"string"}

Image2 = 'D.jpeg' #@param {type:"string"}

img1 = load_and_resize(Image1)
img2 = load_and_resize(Image2)

images_resized = tf.concat([img1, img2], axis=0)

plt.figure(figsize=(10, 6))
for idx in (0, 1):
    plt.subplot(1, 2, idx + 1)
    plt.imshow(images_resized[idx] / 255)
    plt.axis("off")

plt.show()


In [ ]:
# Preprocess the sample images in a specific way that the model is expecting
# Call method: resnet.preprocess_input()

inputs = applications.resnet50.preprocess_input(images_resized)

Y_proba = modelR.predict(inputs)
Y_proba.shape

top_K = applications.resnet50.decode_predictions(Y_proba, top=3)
for image_index in range(len(images_resized)):
    print(f"Image #{image_index}")
    for class_id, name, y_proba in top_K[image_index]:
        print(f"  {class_id} - {name:12s} {y_proba:.2%}")

**2. RESNET for FineTuning - The Flowers Dataset**

**2.1. Data Fetching and Loading**

In [ ]:
# Download the Flowers dataset: https://www.kaggle.com/datasets/imsparsh/flowers-dataset

# Create folder flower_photos
# Inside this folder there are 5 subfolders, one for each category

!curl -O https://storage.googleapis.com/download.tensorflow.org/example_images/flower_photos.tgz

!tar -xzvf flower_photos.tgz

!rm flower_photos.tgz


In [ ]:
# The images in the original folders are not divided in train and validation datasets
# The following code divides samples into 80% training and 20% validation. No test set is created

# https://www.tensorflow.org/api_docs/python/tf/data/Dataset
# https://www.tensorflow.org/guide/data

# The method image_dataset_from_directory() creates Dataset objects from images located in a specified directory
# https://keras.io/api/data_loading/image/

# Parameters subset and validation_split enable the creation of two datasets (Train: 80%; Validation: 20%)
# This method may shuffle images, adjust size and define the batch size
# This way the dataset is (almost) ready to be processed by the neural network


import pathlib

os.chdir('flower_photos')
path = os.getcwd()
data_dir = pathlib.Path(path)


batch_size = 32
IMG_SIZE = (180, 180)

train_ds, val_ds = keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="both",
    seed=123,
    image_size=IMG_SIZE,
    batch_size=batch_size,
)

class_names = train_ds.class_names


train_ds = train_ds.cache().prefetch(1)
val_ds = val_ds.cache().prefetch(1)

In [ ]:
# Dataset detailed information

print('Nr. of classes: ', len(class_names))
print('Classes: ', class_names, '\n')

# Cardinality
print('Cardinalidade Treino: ', train_ds.cardinality().numpy())
print('Cardinalidade Validacão: ', val_ds.cardinality().numpy(), '\n')

for image_batch, labels_batch in train_ds:
    print('Tensor Batch Image: ', image_batch.shape)
    print('Tensor Batch Label: ', labels_batch.shape)
    break

**2.2. Transfer Learning Model**

In [ ]:

keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

# Load ResNet50 from Keras Applications
# The loaded model does not have the classification head

ResNet_base = keras.applications.resnet50.ResNet50(weights="imagenet", include_top=False, input_shape=(224,224,3))
ResNet_base.trainable = False

# Create the complete model
# Preprocess the images to meet what is expected by ResNet
# Add GlobalAveragePooling + Classification Head

inputs = keras.Input(shape=(180, 180, 3))

x = layers.Resizing(224, 224)(inputs)

x = keras.applications.resnet.preprocess_input(x)
x = ResNet_base(x, training=False)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.5)(x)
x = layers.Dense(128, activation='relu')(x)
outputs = layers.Dense(5, activation='softmax')(x)


model_TL1 = tf.keras.Model(inputs, outputs)

model_TL1.summary()

In [ ]:
# Model Compilation

optimizer = tf.keras.optimizers.Adam(epsilon=0.01)

model_TL1.compile(
    optimizer=optimizer,
    loss = 'sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

In [ ]:
history = model_TL1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=20,
)

In [ ]:
# Visualization of Results

history_frame = pd.DataFrame(history.history)
history_frame.loc[:, ['accuracy', 'val_accuracy']].plot()

In [ ]:
# Top layers are well-trained and now we can unfreeze the weights of the base

ResNet_base.trainable = True

# Compile again

optimizer = tf.keras.optimizers.Adam(epsilon=0.01)

model_TL1.compile(
    optimizer=optimizer,
    loss = 'sparse_categorical_crossentropy',
    metrics=['accuracy'],
)

model_TL1.summary()

In [ ]:
# Resume training.
# It will take a while, even with a GPU.

history = model_TL1.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10,
)

In [ ]:
# Visualization of Results

history_frame = pd.DataFrame(history.history)
history_frame.loc[:, ['accuracy', 'val_accuracy']].plot()

In [ ]:
# We'll consider the predictions on the validation dataset

from sklearn.metrics import confusion_matrix

# True labels
y_true = np.concatenate([y for x, y in val_ds], axis=0)

# Predictions
y_pred_probs = model_TL1.predict(val_ds)
y_pred = np.argmax(y_pred_probs, axis=1)


class_names = ['daisy', 'dandelion', 'rose', 'sunflower', 'tulip']

# Confusion Matrix
cm = confusion_matrix(y_true, y_pred)

cm_percent = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis] * 100

# Plot
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm_percent, interpolation='nearest', cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)

ax.set(xticks=np.arange(len(class_names)),
       yticks=np.arange(len(class_names)),
       xticklabels=class_names, yticklabels=class_names,
       ylabel='Real',
       xlabel='Predicted',
       title='Confusion Matrix (%)')

plt.setp(ax.get_xticklabels(), rotation=45, ha="right", rotation_mode="anchor")

for i in range(cm_percent.shape[0]):
    for j in range(cm_percent.shape[1]):
        text = ax.text(j, i, f"{cm_percent[i, j]:.1f}%",
                       ha="center", va="center", color="white" if cm_percent[i, j] > 50 else "black")

plt.tight_layout()
plt.show()

**3. EfficientNetB0 for the Cats and Dogs Dataset**

In [ ]:
# Download the Dataset and create a directory PetImages with two folders: Cat and Dog
# https://www.kaggle.com/datasets/tongpython/cat-and-dog
# https://www.microsoft.com/en-us/download/details.aspx?id=54765
# Each of the folders contains images from one class

import os

!curl -O https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5340.zip

!unzip -q kagglecatsanddogs_5340.zip

!rm kagglecatsanddogs_5340.zip

num_skipped = 0
for folder_name in ("Cat", "Dog"):
    folder_path = os.path.join("PetImages", folder_name)
    for fname in os.listdir(folder_path):
        fpath = os.path.join(folder_path, fname)
        try:
            fobj = open(fpath, "rb")
            is_jfif = b"JFIF" in fobj.peek(10)
        finally:
            fobj.close()

        if not is_jfif:
            num_skipped += 1
            # Delete corrupted image
            os.remove(fpath)

print(f"Deleted {num_skipped} images.")


In [ ]:
# Shape of Dataset

cat_dir = 'PetImages/Cat'
dog_dir = 'PetImages/Dog'

num_cats = len(os.listdir(cat_dir))
num_dogs = len(os.listdir(dog_dir))

print('Cat images:', num_cats)
print('Dog images: ', num_dogs)

In [ ]:
# Add Preprocessing




In [ ]:
keras.backend.clear_session()
tf.random.set_seed(42)
np.random.seed(42)

# Load EfficientNetB0 from Keras Applications
# The loaded model does not have the classification head

ENet_base = keras.applications.EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224,224,3))
ENet_base.trainable = False

# Create the complete model


# Compile the model



In [ ]:
# Train the model




In [ ]:
# Analyze results



In [ ]:
# Unfreeze the backbone

ENet_base.trainable = True

# Compile again



In [ ]:
# Resume training



In [ ]:
# Analyze results

